**Imports and Setup**

In [ ]:
import cv2
import mediapipe as mp
import os
import csv
import numpy as np
import time

In [ ]:
# variables to be modified, change this per data collection
setup_num = 5

VIDEO_PATH = f'..\\Video Dataset\\setup_{setup_num}\\face_setup_{setup_num}.mp4'
CSV_PATH   = f'..\\CSV Dataset\\landmarks_setup_{setup_num}.csv'

**Project Setup and Verfication**

In [ ]:
mp_face_mesh = mp.solutions.face_mesh # Holistic model
mp_drawing   = mp.solutions.drawing_utils # Drawing utilities

In [ ]:
# load video
cap = cv2.VideoCapture(VIDEO_PATH)

frame_count = 0
frames_to_skip = 1 # change this to skip frames, used for faster processing but less accurate (1 = no skip)

with mp_face_mesh.FaceMesh(min_detection_confidence=0.5, min_tracking_confidence=0.5, max_num_faces=1) as face_mesh:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # skip every other frame
        frame_count += 1
        if frame_count % frames_to_skip != 0:
            continue

        # recolor for MediaPipe
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = face_mesh.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # print results, for troubleshooting
        # print(results.multi_face_landmarks)

        # draw face mesh
        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0]
            mp_drawing.draw_landmarks(
                image,
                face_landmarks,
                mp_face_mesh.FACEMESH_TESSELATION,
                mp_drawing.DrawingSpec(color=(80, 110, 10),  thickness=1, circle_radius=1),
                mp_drawing.DrawingSpec(color=(80, 256, 121), thickness=1, circle_radius=1)
            )

        cv2.imshow('Setup Test', image)
        if cv2.waitKey(1) & 0xFF == ord(' '):
            break

cap.release()
cv2.destroyAllWindows()

**Create CSV File with Headers**

In [ ]:
feature_column_count = len(results.multi_face_landmarks[0].landmark)

header_columns = ['class', 'time']

for i in range(1, feature_column_count + 1):
    header_columns.append('x' + str(i))
    header_columns.append('y' + str(i))
    header_columns.append('z' + str(i))

# print header row
print(header_columns)

In [ ]:
print(len(header_columns))

In [ ]:
# save data
with open(CSV_PATH, 'w', newline='') as file:
    writer = csv.writer(file, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(header_columns)

**Collect Data from Video and Save to CSV File**

In [ ]:
# load video
cap = cv2.VideoCapture(VIDEO_PATH)

class_name = ['placeholder']

frame_count = 0
current_frame = 1
frames_to_skip = 1 # change this to skip frames, used for faster processing but less accurate (1 = no skip)

with mp_face_mesh.FaceMesh(min_detection_confidence=0.5, min_tracking_confidence=0.5, max_num_faces=1) as face_mesh:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # skip every other frame
        frame_count += 1
        if frame_count % frames_to_skip != 0:
            continue

        # recolor for MediaPipe
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = face_mesh.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # print results, for troubleshooting
        # print(results.multi_face_landmarks)

        # draw face mesh
        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0]
            mp_drawing.draw_landmarks(
                image,
                face_landmarks,
                mp_face_mesh.FACEMESH_TESSELATION,
                mp_drawing.DrawingSpec(color=(80, 110, 10),  thickness=1, circle_radius=1),
                mp_drawing.DrawingSpec(color=(80, 256, 121), thickness=1, circle_radius=1)
            )

        try:
            frame_time = [(f"{((frame_count / 30) // 60):.0f}:{((frame_count / 30) % 60):.0f}")]
            face = results.multi_face_landmarks[0].landmark
            face_row = np.array([[landmark.x, landmark.y, landmark.z] for landmark in face]).flatten().tolist()
            row = class_name + frame_time + face_row

            with open(CSV_PATH, 'a', newline='') as file:
                writer = csv.writer(file, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                writer.writerow(row)

        except Exception as e:
            print(e)

        # print frames
        print("\rFrame " + str(current_frame) + " saved", end="", flush=True)
        current_frame += 1

        cv2.imshow('Data Collection', image)
        if cv2.waitKey(1) & 0xFF == ord(' '):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
print(len(row))

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(df.shape)